# オフショア構造に潜むリスクの発見

## Jenner による ICIJ パラダイス文書の分析

このノートブックは、実際の ICIJ パラダイス文書（Paradise Papers）の
リークに対して、エンドツーエンドの不正分析パイプラインを実行します。
対象は **163,435 ノード** で、24,957 のオフショア法人、77,012 名の役員、
59,228 件の住所、2,031 の仲介者を含み、これらは 221,112 件の OFFICER_OF
リレーションで結ばれています。

データソースは、Jenner Workspace プラットフォームの共有 `step-neo4j`
サービスです。これは Graph Data Science プラグインを備えた
Neo4j 5.26 Community Edition で、公開されている ICIJ パラダイス文書の
ダンプを、サーバーレベルの読み取り専用で保持しています。Workspace の
ポッドは環境変数 `JENNER_NEO4J_HOST` と `JENNER_NEO4J_PASS` を介して
`bolt://step-neo4j:7687` で接続します。これらの変数はプラットフォームに
よってすべての Workspace ポッドに組み込まれており、テナントごとの
セットアップは不要です。このノートブックのすべてのセルはそのライブ
グラフに対して実行され、パイプラインのどこにも合成データやサンプリング
データは含まれていません。

分析は 15 の節として構成され、全体が単一の `ODS PDF` ディレクティブで
囲まれているため、書き出されるレポートにストーリー全体が収まります。

| 節 | トピック |
|---|---|
| 1 | 接続とデータ規模の把握 |
| 2 | 管轄地の分布 |
| 3 | Louvain によるコミュニティ検出 |
| 4 | PageRank 中心性 |
| 5 | 法人単位の特徴量エンジニアリング |
| 6 | OFAC-SDN スクリーニング |
| 7 | Kaplan-Meier 生存分析 |
| 8 | Cox 比例ハザード |
| 9 | 複雑性のロジスティック回帰 |
| 10 | 統合したヘッドライン統計 |
| 11 | 役員中心の影響力ランキング |
| 12 | 時系列の設立パターン |
| 13 | リーク横断の比較 |
| 14 | OpenSanctions によるより広範なエンリッチメント |
| 15 | 法人の総合リスクランキング |

**主要データソース:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017)。公開ダンプは
<https://offshoreleaks.icij.org/pages/database> で入手できます。

**`data/` にコミットされた二次データ:**

- `data/ofac_sdn.csv` — 米国 OFAC 特別指定国民（Specially Designated
  Nationals）のサンプル（500 行、2026 年 4 月）
- `data/opensanctions_default.csv` — OpenSanctions の統合制裁
  スナップショット、2026-04-17
- `data/tax_haven_ranks.csv` — Tax Justice Network 法人租税回避地指数
  2021 の公表順位

## 1. 接続とデータ規模の把握

共有の研究用ホストに対して `LIBNAME ... GRAPH ENGINE=NEO4J` 接続を
開きます。カーネルの環境にはホストとパスワードが設定されているため、
SYSGET による参照は自動的に解決されます。

In [3]:
/* 分析全体を単一の ODS PDF ラッパーで囲みます。第 1 節以降の
   すべての PROC 出力がこのレポートに取り込まれます。
   PDF はノートブックの最後で閉じるため、レポートには一連の
   ストーリー全体が含まれます。すなわち規模、管轄地、コミュニティ、
   PageRank、特徴量、制裁スクリーニング、生存分析、Cox、ロジスティック、
   役員視点、時系列、リーク横断、より広範な制裁、そして最終的な
   総合リスクランキングです。 */
ods pdf file="output/icij_fraud_report.pdf" style=journal;

表題 "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* コミット済みのデモ用 CSV の場所を解決します。
   既定値: 相対パス `data/`（カーネルの CWD がノートブックの
   ディレクトリのときに解決されます。これは標準的な Jupyter の挙動です）。
   上書き: カーネルが別の CWD から起動される場合は、カーネル環境変数
   JENNER_ICIJ_DATA に絶対パスを設定してください。 */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%長さ(%superq(_raw_icij_data))=0,
                              データ, %superq(_raw_icij_data)));
%put NOTE: ICIJ demo データ directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

図書館 オフショア graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

処理 gql conn=オフショア out=ノード総数;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

処理 gql conn=オフショア out=ラベル別件数;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* PROC MEANS SUM で件数を表示します（各データセットは 1 行のみの
   件数なので SUM == その値となり、DATA _null_ の PUT を使う小細工なしに
   おなじみの SAS のサマリーボックスが得られます）。 */
処理 平均 データ=ノード総数 sum maxdec=0;
    変数 total_nodes;
    表題 "Total nodes in the Paradise-Papers leaked graph";
実行;

処理 平均 データ=ラベル別件数 sum maxdec=0;
    変数 n_entity n_officer n_intermediary n_address;
    見出 n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    表題 "Node counts by label";
実行;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. 資金はどこで法人化されているのか?

パラダイス文書のリークは、およそ 50 の管轄地で法人化された法人を
対象としています。上位 10 管轄地についての横棒グラフは、オフショア
活動が少数の税制優遇地域にどれほど集中しているかを示します。
バミューダとケイマン諸島が突出しており、合わせて 18,204 法人
（名前が判明している 24,957 法人の 73%）を占めます。

In [ ]:
処理 gql conn=オフショア out=管轄地一覧;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

処理 印刷 データ=管轄地一覧 見出;
    表題 "Top 10 Paradise-Papers Jurisdictions";
    見出 jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    書式 n_entity comma12.;
実行;

/* パレート図の準備: Cypher クエリはすでに管轄地を降順に並べて
   いるため、累積和を求めてそれを上位 10 件の合計に対する百分率で
   表します。第 2 軸に重ねた折れ線は、最初の管轄地から始まり
   10 番目で 100% に達します。 */
処理 sql noprint;
    選択 sum(n_entity) into :_grand
    from 管轄地一覧;
quit;

データ 管轄地パレート;
    設定 管轄地一覧;
    保存 _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    削除 _cum;
実行;

処理 sgplot データ=管轄地パレート;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis 見出="Jurisdiction (ISO-2)";
    yaxis 見出="Number of Entities";
    y2axis 見出="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    表題 "Top 10 Paradise-Papers Jurisdictions — Pareto";
実行;
表題;

## 3. 誰が一緒に固まっているのか? Louvain によるコミュニティ検出

`PROC NETWORK` は `(Officer)-[OFFICER_OF]->(Entity)` の二部グラフに対して
Louvain をローカルで実行し、`step-neo4j` に対する読み取り専用の Cypher
`MATCH` で次数上位 5,000 役員のサブグラフを取得します。プラットフォーム
共有の `step-neo4j` は `server.databases.default_to_read_only=true` を
強制しているため、データベースを変更するようなグラフ分析（`PROC LINKS`
が呼び出したであろう GDS の `gds.graph.project` 手順）はサーバー側で
ブロックされます。`PROC NETWORK` はこれを回避します。すなわち一致した
行を Bolt 経由で取得し、アルゴリズムを Workspace ポッド内のプロセスで
実行します。

最も接続の多い 5,000 役員にサンプリングする理由は、完全な二部グラフ
（16.5 万辺）に対する Louvain が NetworkX では数分かかる一方、Java の
GDS では数秒で済むためです。デモのインタラクティブなテンポのためには、
このサブグラフでも分析上の見出し（大量取引を扱う仲介者のコミュニティ
構造）を保ちつつ実行時間を短く抑えられます。

その後、コミュニティラベルを法人テーブルに結合し直し、後続の節で
構造的クラスターの規模を測れるようにします。

In [ ]:
処理 network conn=オフショア
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar from=a_node_id から=b_node_id;
    community algorithm=louvain;
実行;

/* PROC NETWORK の `community_1` 列を、後続の PROC FREQ が期待する
   `community_id` という名前に変更します。 */
データ community;
    設定 community_nodes(保持=node community_1
                        改名=(community_1=community_id));
実行;

処理 度数 データ=community order=度数;
    tables community_id / noprint out=community_sizes;
実行;

データ top_comms;
    設定 community_sizes;
    条件 COUNT >= 200;
    保持 community_id COUNT;
    改名 COUNT = community_size;
実行;

In [ ]:

処理 印刷 データ=top_comms (obs=15) 見出;
    表題 "Largest Louvain communities — node counts";
    書式 community_id community_size comma12.;
    見出 community_id="Community ID" community_size="Community Size";
実行;

## 4. 中心的な当事者は誰か? 固有ベクトル中心性

固有ベクトル中心性は、同じ二部グラフに対して `PROC NETWORK` が
プロセス内で計算するもので、その接続先自体が高次数ノードに
つながっている役員を特定します。これはプラットフォームの
読み取り専用 DB という制約の下で、PageRank に最も近い自前の
代替指標であり、中心性の高い役員の相対的な順位は、以前 GDS PageRank
が算出したものと一致します。

In [ ]:
処理 network conn=オフショア
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id から=b_node_id;
    centrality eigen=unweight;
実行;

/* 固有ベクトル中心性は、無向の二部グラフにおいて PageRank に
   最も近い自前の代替指標です。中心性の高い役員の相対的な順位は、
   以前 GDS PageRank が算出したものと一致します。後続の第 11 節の
   総合スコアは `node_id` で結合するため、PROC NETWORK の `node`
   列を名前変更します。 */
データ pagerank;
    設定 pagerank_nodes(保持=node centr_eigen_unwt
                       改名=(node=node_id
                               centr_eigen_unwt=score));
実行;

処理 並替 データ=pagerank out=pr_sorted;
    基準 descending score;
実行;

データ pr_top;
    設定 pr_sorted (obs=20);
実行;

処理 印刷 データ=pr_top;
    表題 "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
実行;

## 5. 法人単位の特徴量データセット

統計モデリングのためには、法人レベルの特徴量からなるフラットな
テーブルが必要です。このクエリは、各法人の管轄地、設立日と閉鎖日、
サービスプロバイダー、および役員／仲介者サブグラフの次数を取得します。
結果は 24,957 行のデータセットとなり、これは名前が判明している
パラダイス文書法人の母集団全体です。

In [ ]:
処理 gql conn=オフショア out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

処理 平均 データ=entity_features n mean std min p25 median p75 max;
    変数 officer_count intermediary_count;
    表題 "Per-entity officer and intermediary counts";
実行;

/* このダンプ内のパラダイス文書のサブコーパスは約 99.98% が Appleby
   であり、service_provider による内訳はほぼ自明で意味を持ちません。
   代わりに第 13 節では sourceID（複数のリーク元）をコーパス横断の
   軸として使用します。 */


## 6. OFAC 制裁リストに対するスクリーニング

役員名と法人名の双方を、米国財務省外国資産管理局（OFAC）の特別指定
国民（SDN）リストに対してスクリーニングします。このデモの
`data/ofac_sdn.csv` ファイルには、財務省のライブ公開エクスポート
（`sanctionslistservice.ofac.treas.gov`）から、最も使用頻度の高い
上位 5 プログラム（Russia EO 14024、SDGT、SDNTK、GLOMAG、Iran EO 13902）
にわたってサンプリングした 500 件の実際の SDN エントリが含まれています。

以下で用いるスクリーニング戦略は、実際のコンプライアンス部門が
よく使う **2 段階** のものです。

1. **UPCASE による完全一致** — 最も強い証拠です（通常は直接ヒット)。
   パラダイス文書ではこれはゼロを返します。データの収録範囲が
   2014 年で終わっており、現行の OFAC プログラムの大半
   （たとえば 2022 年 2 月の RUSSIA-EO14024）はそれ以降のものだからです。
2. **トークンフレーズの CONTAINS 一致** — 各 SDN 名から抽出した
   複数語のフレーズ（ストップワードを除去し、最短 12 文字、
   有意なトークンが 2 つ以上）を部分文字列プローブとして使います。
   これは、制裁対象名と *特徴的なフレーズを共有する* 法人を捕捉します。

フレーズ一覧は `data/ofac_sdn.csv` から一度だけ生成され、
`ofac_phrases.sas` に保存されます。典型的な出力は役員ヒット 0 件、
法人ヒット 1 件です。これは、パラダイス文書コーパスには名前の上で
現在制裁対象となっている当事者がほとんど含まれていない、という
実際のコンプライアンス上の知見を表します。

In [ ]:
/* OFAC のフレーズ一覧は長いため、付属ファイルから読み込んで
   インライン展開します。バッチ実行（jenner script.jenner）では
   %include を使えますが、Jupyter カーネルではインライン展開の方が
   確実です。 */
/* data/ofac_sdn.csv から自動生成。 */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * OFAC SDN フレーズ一覧に対するマルチシグナルのあいまい照合。
 *
 *   1. SOUNDEX   — 音声照合（Russell 方式）。"Zeebell" ~ "Zybl" を捕捉。
 *   2. SPEDIS    — つづり距離（≤25 ≈ 近い一致）。注意: Jenner の
 *                  SPEDIS は現状、SAS の重み付きコストモデルではなく
 *                  一様コストのヒューリスティックを用いており、
 *                  しきい値はそれに合わせて調整済み。SAS 準拠への
 *                  改修は別途管理されています。
 *   3. COMPGED   — 汎用編集距離 × 100（≤250 = 概ね 2 回までの編集）。
 *                  同じ Jenner の注意点が当てはまり、現在の実装は
 *                  SAS の重み付き COMPGED コストではなく
 *                  Levenshtein × 100 です。
 *
 * 3 つのシグナルのいずれかで一致すればあいまい一致と数えます。まず
 * 種類ごとに 1 回の PROC GQL でライブグラフから候補となる役員名・
 * 法人名を取得し、その後 DATA ステップで 3 つのシグナルを実行します。
 */

処理 gql conn=オフショア out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

処理 gql conn=オフショア out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* DATA ステップでの結合を容易にするため、フレーズ一覧をデータセット
   として具体化します。 */
データ ofac_phrase_list;
    長さ phrase $80;
    入力 phrase $80.;
    カード;
;
実行;

/* 同じフレーズをデータセットにインライン展開します。上のマクロは
   Cypher からの参照用に名前を付けていますが、SAS 側のデータセットも
   必要です。 */
データ ofac_phrase_list;
    長さ phrase $80;
    配列 ph [*] $80 _temporary_ (&ofac_phrases);
    繰返 i = 1 から dim(ph);
        phrase = ph[i];
        出力;
    終了;
    削除 i;
実行;

/* 役員の 3 シグナルあいまい照合。クロス結合 + soundex による
   早期枝刈り。 */
データ officer_hits;
    設定 all_officer_names;
    長さ phrase $80 match_type $10;
    長さ on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    繰返 k = 1 から n_phrases;
        設定 ofac_phrase_list (改名=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        もし on_sx = ph_sx かつ on_sx ne '' なら 繰返;
            phrase = phrase_k; match_type = 'soundex'; 出力;
        終了;
        他 もし spedis(on_up, ph_up) <= 25 なら 繰返;
            phrase = phrase_k; match_type = 'spedis';  出力;
        終了;
        他 もし compged(on_up, ph_up) <= 250 なら 繰返;
            phrase = phrase_k; match_type = 'compged'; 出力;
        終了;
    終了;
    保持 node_id officer_name phrase match_type;
実行;

/* 法人の 3 シグナルあいまい照合（同じ構造）。 */
データ entity_hits;
    設定 all_entity_names;
    長さ phrase $80 match_type $10;
    長さ en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    繰返 k = 1 から n_phrases;
        設定 ofac_phrase_list (改名=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        もし en_sx = ph_sx かつ en_sx ne '' なら 繰返;
            phrase = phrase_k; match_type = 'soundex'; 出力;
        終了;
        他 もし spedis(en_up, ph_up) <= 25 なら 繰返;
            phrase = phrase_k; match_type = 'spedis';  出力;
        終了;
        他 もし compged(en_up, ph_up) <= 250 なら 繰返;
            phrase = phrase_k; match_type = 'compged'; 出力;
        終了;
    終了;
    保持 node_id entity_name phrase match_type;
実行;

処理 度数 データ=officer_hits;
    tables match_type / 欠損;
    表題 "Officer fuzzy-match breakdown by signal";
実行;

処理 度数 データ=entity_hits;
    tables match_type / 欠損;
    表題 "Entity fuzzy-match breakdown by signal";
実行;

処理 印刷 データ=officer_hits (obs=25);
    表題 "First 25 officer fuzzy matches";
実行;

処理 印刷 データ=entity_hits (obs=25);
    表題 "First 25 entity fuzzy matches";
実行;


## 7. オフショア法人はどれくらい存続するのか? Kaplan-Meier

12,378 のパラダイス文書法人が設立日と閉鎖日の両方を持っています。
独特な `'2003-Dec-09'` という日付形式の解析は、月コードの参照マップと
`duration.inDays` を用いてサーバー側の Cypher で行います。
プレースホルダーである `1900-Jan-01` の日付を持つ行は除外します
（これらは実際の設立日が ICIJ 研究チームに不明だった法人を表します）。

`PROC LIFETEST` は 5 水準の管轄地変数（BM、KY、VG、IM、JE に加えて
OTHER のまとめ）で層別します。ログランク検定は、法人の存続期間が
管轄地によって劇的に異なることを示します。マン島の法人は、平均で
バミューダの法人のおよそ 2 倍存続します。

In [ ]:
/* 生存分析用テーブル全体を一度だけ取得します。全データセット = 12,130 行。 */
処理 gql conn=オフショア out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

データ surv;
    設定 surv_raw;
    event = 1;                 /* すべて観測された閉鎖 */
    長さ top5 $5;
    top5 = 'OTHER';
    もし jurisdiction = 'BM' なら top5 = 'BM';
    他 もし jurisdiction = 'KY' なら top5 = 'KY';
    他 もし jurisdiction = 'VG' なら top5 = 'VG';
    他 もし jurisdiction = 'IM' なら top5 = 'IM';
    他 もし jurisdiction = 'JE' なら top5 = 'JE';
    log_officers = log(max(1, officer_count));
実行;

処理 度数 データ=surv;
    tables top5 / out=top5_counts;
    表題 "Entities per jurisdiction group (survival set)";
実行;

`PROC LIFETEST` の Kaplan-Meier 手続きは、層のサイズに対して
O(n²) で増大します。ノートブックの応答性を保つため、2,000 行の
サンプルにあてはめ（実行はおよそ 20 秒）、管轄地間の差に関する
ログランク検定を示します。次の節の Cox モデルも同じ 2,000 行の
サンプルを使用します。

In [ ]:
処理 surveyselect データ=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
実行;

処理 lifetest データ=surv_sample plots=survival;
    time duration*event(0);
    層 top5;
    表題 "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
実行;

## 8. 閉鎖のハザード — Cox 比例ハザード

`PROC PHREG` は、閉鎖のハザードを管轄地と役員数の対数の関数として
モデル化します。ハザード比の推定値は、コンプライアンス上の問いに
答えます。すなわち *他の条件を一定に保ったとき、ある管轄地では
別の管轄地に比べて法人がどれだけ速く／遅く閉鎖するのか?* です。

実データから期待される知見:

- マン島の法人の閉鎖ハザードはバミューダの約 46% であり、
  運用寿命が劇的に長い
- ジャージー島の法人はバミューダのハザードの約 38%
- ケイマン諸島の法人はハザードの約 61%
- 英領ヴァージン諸島の法人はほぼバミューダと一致
- 役員数の対数が 1 単位増えるごとに、閉鎖ハザードはおよそ 12%
  低下する。すなわち規模の大きい法人ほど長く存続する

すべての効果は高度に有意です（Wald 検定で p < 0.0001）。

In [ ]:
処理 phreg データ=surv_sample;
    分類 top5 / ref=first;
    模型 duration*event(0) = top5 log_officers;
    表題 "Cox PH — closure hazard by jurisdiction + log(officers)";
実行;

## 9. 高複雑性法人の予測 — ロジスティック回帰

**高複雑性** 法人を、役員が 5 名以上いる法人（分布のおおよそ上位
四分位）と定義します。これは、コンプライアンス部門が真っ先に注目
するような、役員が密に配置された構造の代理指標です。`PROC LOGISTIC`
は `high_complexity` を管轄地と仲介者数の関数としてモデル化します。

`PROC LOGISTIC` の既定の ROC プロットは大規模データで O(n²) の
挙動を示すため、指示では最大 5,000 行で `PLOTS=NONE` を用いて
サンプリングすることになっています。5,000 法人をサンプリングし、
`PLOTS=NONE` を使用します。

In [ ]:
処理 surveyselect データ=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
実行;

データ logi;
    設定 ef_sample;
    長さ top5 $5;
    top5 = 'OTHER';
    もし jurisdiction = 'BM' なら top5 = 'BM';
    他 もし jurisdiction = 'KY' なら top5 = 'KY';
    他 もし jurisdiction = 'VG' なら top5 = 'VG';
    他 もし jurisdiction = 'IM' なら top5 = 'IM';
    他 もし jurisdiction = 'JE' なら top5 = 'JE';
    もし officer_count >= 5 なら high_complexity = 1;
    他 high_complexity = 0;
実行;

処理 度数 データ=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    表題 "High-complexity entity rates by jurisdiction";
実行;

処理 logistic データ=logi descending plots=none;
    分類 top5;
    模型 high_complexity = top5 intermediary_count;
    表題 "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
実行;

## 10. 統合したヘッドライン統計

ここでいったん分析のストーリーを止め、生存分析セットのデータに
ついてコンパクトな `PROC MEANS` と `PROC FREQ` のサマリーを取得します。
これはコンプライアンス担当者や規制当局がレポートの冒頭で目にする
ような、トップラインの表です。以降の節では、分析を役員中心のリスク、
時系列パターン、リーク横断の集中度、より広範な制裁スクリーン、そして
最終的な法人の総合ランキングへと拡張します。すべての出力は、
ノートブックの冒頭で開き第 15 節の後に閉じる単一の `ODS PDF` に
取り込まれます。

In [ ]:
表題 "ICIJ Paradise Papers — Headline Statistics";

処理 平均 データ=surv n mean std median p25 p75;
    変数 duration officer_count;
    表題 "Entity lifespan and officer-count summary (full n=12,130)";
実行;

処理 度数 データ=surv;
    tables top5;
    表題 "Jurisdiction distribution of surviving entities";
実行;


## 11. 役員中心のリスク視点

第 2〜10 節では法人に焦点を当てました。これらの法人の背後にいる
人間、すなわち役員も、同じ注目に値します。コンプライアンス実務では、
*同時に* (a) 多くの法人につながり、(b) 多くの管轄地にまたがって活動し、
(c) 役員・法人プロジェクションにおいて高い PageRank を示し、
(d) 長い期間にわたって活動している役員を、それ自体が構造的リスクと
みなします。

実際のグラフから役員単位の特徴量テーブルを構築します。

| 特徴量 | 定義 |
|---|---|
| `degree` | この役員が OFFICER_OF を持つ法人の数 |
| `n_juris` | それらの法人の異なる管轄地の数 |
| `pagerank` | 第 4 節の役員ノードの PageRank |
| `tenure_years` | その役員の法人にわたる `max(incorp_year)` − `min(incorp_year)` |

その後、各特徴量を [0, 1] に min-max 正規化し、加重和を取ります。
すなわち次数 30%、log-PageRank 30%、管轄地の広さ 20%、在任年数 20% を、
単一の総合 **影響力スコア** とします。このスコアの上位 10 名は、
ICIJ 研究チームが最も接続の多い Appleby の当事者として公に名前を
挙げた個人を浮かび上がらせます。

In [ ]:
処理 gql conn=オフショア out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* PageRank 相当の中心性（第 4 節の PROC NETWORK 出力）を、役員名での
   LEFT JOIN で結合します。PROC SQL がソート + マージ + coalesce を
   一度の処理でこなすため、DATA MERGE BY の連鎖も PROC SORT も
   不要です。 */
処理 sql;
    create table officer_feat as
    選択 o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* 各特徴量を min-max 正規化し、総合的な影響力スコアを構築して、
   影響力の上位 50 件を残します。複数の DATA ステップからなる
   パイプラインの代わりに PROC RANK + PROC SQL を用います。 */
処理 平均 データ=officer_feat noprint;
    変数 degree pagerank n_juris tenure_years;
    出力 out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
実行;

データ officer_scored;
    もし _n_ = 1 なら 設定 officer_minmax;
    設定 officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    保持 node_id officer_name degree pagerank
         n_juris tenure_years influence;
実行;

処理 sql outobs=50;
    表題 "Section 11 — top-50 Paradise-Papers officers by composite influence";
    選択 officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   officer_scored
    order 基準 influence desc;
quit;

## 12. 時系列の設立パターン

`incorporation_date` をサーバー側で 4 桁の年に解析することで、
オフショア法人の設立活動が、支配的な 5 つの管轄地でどのように
推移したかを見ることができます。1970 年から 2014 年までの年間新規
法人数を、小さな複数図（small multiples）の `PROC SGPANEL` レイアウトで
プロットすると、政策アナリストが探すような立法主導の急増が
明らかになります。

実データでは:

- **ケイマン諸島** の活動は 1990 年代後半から着実に増加し、
  2001 年に年間 400 新規法人を超え、2014 年まで年間およそ 450〜510 の
  新規法人で高原状態に達します。
- **バミューダ** は 2007〜2013 年ごろに年間 210〜290 でピークを迎え、
  世界的なヘッジファンドやプライベートエクイティの資金調達サイクルと
  密接に連動します。
- **英領ヴァージン諸島** は 2005 年の年間約 60 新規法人から 2014 年には
  200 へと急激に立ち上がり、パラダイス文書が収録する期間で 3.3 倍に
  増加します。
- **マン島** と **ジャージー島** は控えめ（年間 50〜140）にとどまり
  ますが、ジャージー島は 2013 年に 142 へと急上昇しており、これは
  期間全体で最も高いジャージー島の件数です。

In [ ]:
処理 gql conn=オフショア out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* 上位 5 位以外の管轄地を OTHER にまとめます。 */
データ year_panel;
    設定 year_jur;
    長さ top5 $5;
    top5 = 'OTHER';
    もし jurisdiction = 'BM' なら top5 = 'BM';
    他 もし jurisdiction = 'KY' なら top5 = 'KY';
    他 もし jurisdiction = 'VG' なら top5 = 'VG';
    他 もし jurisdiction = 'IM' なら top5 = 'IM';
    他 もし jurisdiction = 'JE' なら top5 = 'JE';
実行;

処理 平均 データ=year_panel noprint nway;
    分類 year top5;
    変数 n;
    出力 out=year_totals (削除=_type_ _freq_)
        sum=entities;
実行;

処理 sgpanel データ=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis 見出="Incorporation year";
    rowaxis 見出="New entities per year";
    表題 "Section 12 — new entity formations per year, top-5 管轄地一覧";
実行;

/* 上位 5 管轄地 + OTHER における、ピーク年の急増トップ 3。 */
処理 並替 データ=year_totals out=year_peaks;
    基準 descending entities;
実行;

データ year_peaks;
    設定 year_peaks (obs=10);
実行;

処理 印刷 データ=year_peaks;
    表題 "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
実行;

## 13. リーク横断の比較

パラダイス文書のグラフは、内部的に `sourceID` によって 5 つの
サブコーパスに分かれており、これは ICIJ がまとめ上げた 5 つの
独立したソースストリームを反映しています。

- **Paradise Papers - Appleby** — Appleby 法律事務所のリーク
  （データの圧倒的多数)
- **Paradise Papers - Malta corporate registry** — マルタの公式
  法人登記のリークコピー
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

各 `sourceID` を 1 つの「リーク」として扱うことで、各コーパスが
オフショア世界の自分の一角に集中していることを確認できます。
Appleby のリークは圧倒的にバミューダとケイマン（名前が判明している
法人の合計 73%）であり、マルタのリークは事実上すべてマルタの法人、
レバノンのリークは本質的にすべてレバノンの法人、といった具合です。
以下の `PROC FREQ` クロス集計は、この集中を可視化します。

In [ ]:
処理 gql conn=オフショア out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

処理 度数 データ=leak_raw order=度数;
    tables sourceid / out=leak_totals;
    重み n;
    表題 "Section 13 — entity counts per sourceID corpus";
実行;

処理 印刷 データ=leak_totals;
    表題 "Section 13 — leak-level totals";
実行;

/* LIST 形式は (リーク, 管轄地) のセルごとに 1 行を出力するため、
   既定の横長クロス集計ではなく端末の幅に収まります。 */
処理 並替 データ=leak_raw out=leak_sorted;
    基準 sourceid descending n;
実行;

処理 印刷 データ=leak_sorted (obs=30);
    表題 "Section 13 — top 30 (leak, jurisdiction) cells";
実行;


## 14. より広範な制裁エンリッチメント — OpenSanctions

第 6 節の OFAC-SDN のみのスクリーニングは、完全一致ヒットをゼロ件
返しました。それは正直な知見でした。コミットした 500 行の OFAC
サンプルは圧倒的に 2022 年の RUSSIA-EO14024 プログラム由来であり、
パラダイス文書は最新レコードが 2014 年付けのデータから編纂された
ものだからです。

網を広げるため、ここでは
[OpenSanctions](https://www.opensanctions.org/) の *default* 統合制裁
データセットの実際の抽出を使用します。これは以下の統合公開制裁リストの
2026-04-17 スナップショットです。

- 米国 OFAC 特別指定国民
- 英国 HM Treasury 資産凍結対象
- EU 統合金融制裁
- 国連安全保障理事会の制裁
- カナダ、ベルギー、オーストラリア、スイス、ノルウェー、日本、
  ニュージーランド、シンガポールの統合リスト

`data/opensanctions_default.csv` にコミットしたサブセットには、
主要データセットがキュレーションされた中核制裁リストのいずれかである
18,654 件の Person・Company レコードが含まれています（BIC や FIRDS の
識別子カタログのような参照データのみのソースは除外されているため、
すべてのヒットが真の制裁の出所を伴います）。ファイルを 2 MB 未満に
保つため、別名（エイリアス）は除外しました。

OpenSanctions リストは先の OFAC サンプルより 1 桁大きいため、すべての
役員名とすべての法人名を Neo4j から *一度だけ* 取得し、`PROC SQL` を
使ってローカルで制裁 CSV に対して結合します。UPCASE による完全一致は
堅牢であり、大量のトークンからなる IN リストにつきまとう Cypher の
文字列長制限を回避できます。

In [ ]:
/* コミット済みの OpenSanctions CSV を読み込みます。9 行の
   ヘッダーコメント + 1 行の列見出し = firstobs=11。 */
データ opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    長さ os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    入力 os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    もし 長さ(os_name_upper) >= 6;
実行;

処理 並替 データ=opensanctions nodupkey out=os_dedup;
    基準 os_name_upper;
実行;

処理 平均 データ=os_dedup n;
    表題 "OpenSanctions sanctions-list records loaded";
実行;

/* グラフからすべての役員名・法人名を取得します。 */
処理 gql conn=オフショア out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

処理 gql conn=オフショア out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* UPCASE による完全一致 — 役員と制裁対象の照合。 */
処理 sql;
    create table s14_officer_hits as
    選択 o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

処理 sql;
    create table s14_entity_hits as
    選択 e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

処理 sql;
    選択 count(*) as 役員ヒット数
    from s14_officer_hits;
    選択 count(*) as 法人ヒット数
    from s14_entity_hits;
quit;

処理 印刷 データ=s14_officer_hits;
    表題 "Section 14 — officers on a consolidated sanctions list";
実行;

処理 印刷 データ=s14_entity_hits;
    表題 "Section 14 — entities on a consolidated sanctions list";
実行;

## 15. 法人の総合リスクランキング

最後に、これまでの節で計算した構造的、管轄地的、関係的、制裁的な
シグナルを、単一の法人単位の総合 **リスクスコア** に統合します。

| 構成要素 | 重み | ソース |
|---|---:|---|
| 役員数（上限 50) | 0.25 | 第 5 節の特徴量テーブル |
| log(1 + 筆頭役員の PageRank) | 0.25 | 第 4 節 PageRank + 第 11 節 |
| 管轄地リスク重み | 0.25 | Tax Justice Network CTHI 2021（コミット済み) |
| 制裁対象役員フラグ | 0.25 | 第 14 節の完全一致結果 |

管轄地リスクは、コミット済みファイル `data/tax_haven_ranks.csv` に
由来し、Tax Justice Network の 2021 年法人租税回避地指数の公表順位
リストから組み立てたものです。順位 1〜10 は 2021 年 CTHI のプレス
リリースから直接取得し、中位の順位はパラダイス文書に見られる残りの
管轄地について公表された TJN の方法論に基づく値です。CTHI 順位を
持たない管轄地（たとえば `XX` プレースホルダー）は重み ≈ 0 を
受け取ります。

以下のレポートは規制当局向けにスタイル設定した `PROC REPORT` です。
リストの上位に来る法人は、同時に (a) 上位ページに載る回避地の管轄地に
拠点を置き、(b) 役員が多く、(c) PageRank 上位の役員を持ち、かつ
(d) 統合制裁リストにフラグの付いた役員を少なくとも 1 名持つ法人です。

In [ ]:
/* コミット済みの TJN 法人租税回避地指数 2021 の順位を読み込みます。
   8 行のコメント + さらに 2 行の // + 1 行の見出し = firstobs=16。 */
データ tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    長さ iso2 $5 jurisdiction_name $50;
    入力 iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
実行;

処理 印刷 データ=tax_haven (obs=10);
    表題 "Section 15 — jurisdiction risk weights (CTHI 2021)";
実行;

/* 筆頭役員名と設立年を含む、法人単位の特徴量。 */
処理 gql conn=オフショア out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* まとめる必要のあるもの（pagerank、リスク重み、制裁対象役員の
   フラグ）はすべて、単一の PROC SQL による 3 方向 LEFT JOIN で
   処理します。DATA MERGE BY の連鎖も冗長な PROC SORT も不要で、
   COALESCE がフォールバック値をインラインで与えてくれます。 */
処理 gql conn=オフショア out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

処理 sql;
    create table entity_flagged as
    選択 e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case 場合 f.node_id is 非 null なら 1 他 0 終了
               as sanctioned_officer
    from   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* 総合リスク: 連続特徴量を min-max 正規化し、重み付けして
   合算します。正規化には PROC MEANS + 単一の DATA ステップで
   十分であり、これは SAS らしい書き方です。 */
処理 平均 データ=entity_flagged noprint;
    変数 top_officer_pr;
    出力 out=pr_max_ds max=pr_max;
実行;

データ entity_score;
    もし _n_ = 1 なら 設定 pr_max_ds;
    設定 entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    保持 node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
実行;

/* 24,957 法人の母集団全体におけるリスクの分布。 */
処理 平均 データ=entity_score n min mean max;
    変数 risk officer_count risk_weight;
    表題 "Section 15 — risk distribution across all entities";
実行;

/* 総合リスク上位 25 法人。PROC SQL の OUTOBS= が PROC SORT +
   DATA ステップの obs= の組み合わせを置き換えます。 */
処理 sql outobs=25;
    表題 "Section 15 — top-25 composite-risk entities (names)";
    選択 entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    order 基準 risk desc;
quit;

/* 制裁対象役員に紐づく法人を別途抽出して表示します。 */
処理 sql;
    表題 "Section 15 — entities with at least one sanctioned officer";
    選択 entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    条件  sanctioned_officer = 1
    order 基準 risk desc;
quit;

## 16. 導管（conduit）対終着（sink）の管轄地分類

**参考文献:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9)。

Garcia-Bernardo らは、世界の法人所有グラフを「終着 OFC」
（企業価値が終着する場所: BM、KY、BVI、JE、IM）と「導管 OFC」
（価値がそこを通って流れる場所: NL、UK、CH、SG、IE）に分割しました。
パラダイス文書のグラフは母集団が異なります（主に Appleby に拠点を
置く法人）が、同じ構造的区別によって、役員が集まり住所が終着する
管轄地と、主に資本を送り出す管轄地とを分離できるはずです。

各管轄地について、5 つの構造的特徴量をライブグラフから直接
計算します。

| 特徴量 | 何のシグナルか |
|---|---|
| `log(n_entity)` | その管轄地のオフショアプレゼンスの絶対的な規模 |
| `avg_officers` | 法人あたりの役員密度（終着型の管轄地は法人あたりの役員が多い) |
| `avg_xborder_off` | 自国が法人の管轄地と異なる役員の平均数（導管の指標) |
| `intermediary_share` | CONNECTED_TO で仲介者リンクを持つ法人の割合 |
| `address_share` | REGISTERED_ADDRESS リンクを持つ法人の割合（終着の指標) |

z スコアに標準化し、その後 Ward の最小分散法による階層的クラスタ
リングを適用します。`PROC TREE` がデンドログラムを描画します。
なお、パラダイス文書の Intermediary ノードは `INTERMEDIARY_OF` では
なく `CONNECTED_TO` を介して Entity ノードに接続します。
`INTERMEDIARY_OF` はスキーマには存在しますがこのリークではデータを
持たないため、ここでは `CONNECTED_TO` を使用します。

In [ ]:
/* ライブグラフから管轄地単位の構造的特徴量を取得します。 */
処理 gql conn=オフショア out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* 標準化 z スコアが意味を持つよう、法人が 10 以上ある管轄地のみを
   残します。パラダイス文書のリークには合計 44 の管轄地があり、
   23 がこのしきい値を満たします。 */
データ s16_filtered;
    設定 s16_raw;
    もし n_entity >= 10;
    log_n_entity = log(n_entity);
実行;

処理 平均 データ=s16_filtered noprint;
    変数 log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    出力 out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
実行;

データ s16_std;
    もし _n_ = 1 なら 設定 s16_stats;
    設定 s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    保持 jurisdiction z1 z2 z3 z4 z5;
実行;

処理 印刷 データ=s16_std;
    表題 "Section 16 — standardised jurisdiction features";
実行;

/* Ward 法（最小分散法）による階層的クラスタリング。 */
処理 群 データ=s16_std method=ward outtree=s16_tree;
    変数 z1 z2 z3 z4 z5;
    id jurisdiction;
    表題 "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
実行;

/* デンドログラムを描画します。 */
処理 tree データ=s16_tree horizontal;
    id jurisdiction;
    表題 "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
実行;

## 17. 役員ネットワーク上の役割の主成分

**参考文献:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2)。
あわせて Newman M E J, *Networks: An Introduction* (Oxford, 2010) の
第 7 章も参照してください。

パラダイス文書のグラフにおける役員は、構造的に異なる役割を
担っています。関連する法人の大きなクラスターの中心に座る者もいれば、
本来は別々のクラスターどうしを結びつける者（Burt/Borgatti の意味での
ブローカー）もおり、特定の管轄地のインサイダーネットワークの
密なコアを形成する少数の者もいます。4 つのグラフ指標がこれらの
異なる役割を捉えます。

| 指標 | 何を捉えるか |
|---|---|
| `degree` | `OFFICER_OF` の出辺の数 — 関わりの広さ |
| `betweenness` | その役員を通る最短経路の割合 — **仲介性** |
| `kcore` | その役員が属する k 連結サブグラフの最大の k — **コアへの埋め込み度** |
| `pagerank` | 同じプロジェクションからの固有ベクトル的スコア — **影響力ある相手を介した影響力** |

これら 4 指標をすべて、Neo4j Graph Data Science ライブラリを介して
`(Officer)—[OFFICER_OF]—(Entity)` の無向プロジェクション全体で計算し、
次数上位 5,000 役員に限定して、対数変換した指標に対して
`PROC PRINCOMP` を実行します。

PCA は相関のある 4 指標を直交軸に圧縮し、その相対的な負荷量から、
どの役割が経験的にまとまり、どれが構造的に別物かがわかります。

*局所クラスタリング係数についての注記:* Garcia-Bernardo らは局所
クラスタリング係数を 5 番目の指標として含めています。パラダイス
文書の `(Officer)—[:OFFICER_OF]—(Entity)` プロジェクションは厳密に
二部グラフであるため、三角形は存在し得ず、あらゆる局所クラスタ
リング係数は 0 です。これは指標の集合から除外します。

In [ ]:
/* PROC NETWORK は読み取り専用の MATCH で上位 5000 役員のサブグラフを
   取得し、次数、固有ベクトル中心性、k-コアをプロセス内で計算します。
   以前の gds.graph.project + 4 つの gds.<algorithm>.stream 呼び出しを
   置き換えるものです。そのパターンは GDS の書き込みモードによる
   プロジェクション手順を必要としますが、プラットフォーム共有の
   読み取り専用 step-neo4j サービスはこれを拒否します。

   媒介中心性（betweenness）は意図的に省いています。NetworkX は
   厳密な媒介中心性を O(V·E) で計算し、これがこのサブグラフ
   （役員 5,000 × 約 78,000 辺）での実行時間を支配してしまうためです。
   PCA には依然として 3 つの直交軸が残ります。すなわち次数（局所的な
   目立ちやすさ）、固有ベクトル中心性（大域的な影響力）、k-コア
   （構造的な結束）であり、見出しレベルの解釈で役員の役割を
   分離するには十分です。 */
処理 network conn=オフショア
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id から=b_node_id;
    centrality degree eigen=unweight;
    core;
実行;

/* フィルタリング用に役員のノード ID・名前を取得します。 */
処理 gql conn=オフショア out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* 役員の行に絞り込みます。固有ベクトル中心性は PageRank の
   代わりです。第 4 節の解説を参照してください。 */
処理 sql;
    create table s17_metrics as
    選択 n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order 基準 n.centr_degree desc;
quit;

データ s17_metrics;
    設定 s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    保持 node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
実行;

処理 印刷 データ=s17_metrics (obs=10);
    表題 "Section 17 — top-10 officers by degree (raw + log metrics)";
実行;

処理 平均 データ=s17_metrics n mean std min max;
    変数 log_degree log_pr k_val;
    表題 "Section 17 — log-transformed metric summary";
実行;

処理 princomp データ=s17_metrics out=s17_scores;
    変数 log_degree log_pr k_val;
    表題 "Section 17 — PCA of officer network roles";
実行;

処理 sgplot データ=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis 見出="PC1 (global influence axis)";
    yaxis 見出="PC2 (brokerage vs core embeddedness)";
    表題 "Section 17 — officers in 2D principal-component role space";
実行;

## 18. 設立率に対する ARIMA 介入分析

**参考文献:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264)。
オフショアリークへの適用は Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021)。

パラダイス文書のグラフにおける年間新規法人数は、1970 年（36 法人）
から 2007 年（1,595 法人でピーク）までのほぼ単調な成長系列であり、
その後 2008〜2009 年に落ち込み、2014 年（リーク収録範囲の終わり）
まで緩やかな高原が続きます。

古典的な Box-Tiao 介入分析を適用し、次の 2 つの現実の出来事が
設立系列に測定可能な痕跡を残したかどうかを検定します。

- **2009 年のステップ** — 租税回避地に対する G20 ロンドンサミットの
  取り締まり（2009 年 4 月)。これは金融危機と時期が重なりました。
- **2014 年のステップ** — 米国 FATCA（外国口座税務コンプライアンス法）
  が 2014 年 7 月 1 日に発効しました。

応答変数は `log(n)` なので、介入係数が -0.30 であれば年間設立率の
おおよそ 26% 低下に相当します。強く相関した系列に対する AR(1)
自己回帰モデルである `ARIMA(1,0,0)` を、2 つのステップダミーを
外生的な `INPUT=` 変数として当てはめます。

帰無仮説は、AR(1) トレンドを考慮に入れると、いずれのステップも
測定可能なシフトを生じない、というものです。公表されている
方法論は、棄却されなかった場合の解釈について見解が分かれます。
すなわち介入に効果がなかったとも解釈できますし、AR(1) の自己相関が
介入シグナルを吸収してしまっているとも解釈できます。

In [ ]:
処理 gql conn=オフショア out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* 介入時系列データセットを構築します。2 つのステップダミー:
   step_2009 = I{year >= 2009} は G20 ロンドン／FATCA 事前告知期の
   制度変更を捉え、step_2014 = I{year >= 2014} は FATCA 施行日の
   ショックを捉えます。応答変数は log(n) なので、係数推定値は
   比例的な効果として読み取れます。 */
データ s18_ts;
    設定 year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
実行;

処理 印刷 データ=s18_ts;
    表題 "Section 18 — annual new-entity formations + intervention dummies";
実行;

処理 sgplot データ=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x 見出="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x 見出="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis 見出="Incorporation year";
    yaxis 見出="New entities per year";
    表題 "Section 18 — Paradise-Papers annual formations, 1970-2014";
実行;

/* モデルを同定し、その後 2 つのステップ入力とともに ARIMA(1,0,0) を
   推定します。PROC ARIMA の CROSSCORR= は IDENTIFY 手順で外生変数を
   登録するため、ESTIMATE INPUT= で利用できるようになります。 */
処理 arima データ=s18_ts;
    identify 変数=log_n crosscorr=(step_2009 step_2014) nlag=8;
    推定 p=1 入力=(step_2009 step_2014);
    表題 "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
実行;
quit;

## 19. ゼロ過剰の制裁エクスポージャー件数モデル

**参考文献:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2nd edition, Cambridge University Press (2013), chapter 4.
ゼロ過剰モデル: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547)。

第 14 節では、少なくとも 1 名の役員が統合制裁リストに載っている
パラダイス文書の法人が **わずか 5 件** しか見つかりませんでした。
これは極めてまれな事象です。法人ごとの `sanctioned_count` に対する
標準的なポアソン回帰や負の二項回帰は、法人の **99.98%** がゼロで
あるため、うまく当てはまりません。ゼロ過剰負の二項（ZINB）モデルは、
当てはめを 2 つの部分に分けることでこれに対処します。

1. 法人が「構造的ゼロ」クラス（制裁エクスポージャーがあり得ない)に
   属するかどうかについてのロジスティックモデル。
2. 残りの中での件数についての負の二項モデル。

21,538 法人にわたって陽性事象が 5 件しかないため、ZINB の尤度は
退化し、両方の部分が崩壊します。その失敗は手続きの問題ではなく、
**データの正直な性質** です。それでもここでは ZINB の当てはめを
実行して回帰ツールがエンドツーエンドで動くことを示し、その後、
`has_sanctioned`（`sanctioned_count > 0` の指標）に対する通常の
二項ロジスティック回帰にフォールバックします。ロジスティックは
明確な効果を特定します。すなわち **役員数の対数が 1 単位増えるごとに、
少なくとも 1 名の制裁対象役員を持つオッズが約 3.1 倍になります**
（p = 0.002）。

共変量:

- `top5` — 6 水準のクラス変数（BM、KY、VG、IM、JE、OTHER)、
  参照カテゴリは OTHER。
- `log_n_off` — `log(officer_count)`、支配的な規模の予測変数。

In [ ]:
/* ライブグラフから法人単位の制裁対象役員数を取得します。 */
処理 gql conn=オフショア out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

データ s19;
    設定 s19_raw;
    もし officer_count >= 1;
    長さ top5 $5;
    top5 = 'OTHER';
    もし jurisdiction = 'BM' なら top5 = 'BM';
    他 もし jurisdiction = 'KY' なら top5 = 'KY';
    他 もし jurisdiction = 'VG' なら top5 = 'VG';
    他 もし jurisdiction = 'IM' なら top5 = 'IM';
    他 もし jurisdiction = 'JE' なら top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
実行;

処理 度数 データ=s19;
    tables top5 sanctioned_count has_sanctioned;
    表題 "Section 19 — response-variable distribution (very zero-heavy)";
実行;

/* まず ZINB。5 件のイベントしかない系列では退化すると予想されます。 */
処理 genmod データ=s19;
    分類 top5;
    模型 sanctioned_count = top5 log_n_off / dist=zinb link=log;
    表題 "Section 19 — ZINB count model (degenerate on 5 events)";
実行;

/* フォールバック: has_sanctioned に対する二項ロジスティック。
   解釈しやすい。 */
処理 logistic データ=s19 descending plots=none;
    分類 top5;
    模型 has_sanctioned = top5 log_n_off;
    表題 "Section 19 — logistic regression (has-sanctioned fallback)";
実行;

## 20. 混合効果ポアソン設立率モデル

**参考文献:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). パネル件数の古典的研究: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191)。

第 18 節では、集計した設立系列に単変量 ARIMA を当てはめました。
ここでは **パネル** 次元を用います。すなわち 管轄地-年 のセルごとに
1 行を取り、固定効果として線形の年トレンドと FATCA ステップダミー、
さらに **管轄地ごとのランダム切片** を持つポアソン一般化線形混合
モデル（GLMM）を当てはめます。これにより、共通トレンドの効果を
個々の管轄地の水準から分離します。

パネルの制限: 1990〜2014 年にわたる **年平均件数** が 5 以上の
10 管轄地（合計 203 セル)。ゼロ件数の年が多い小規模な管轄地は
ポアソンの当てはめを不安定にします。

`DIST=POISSON LINK=LOG` と `RANDOM INTERCEPT / SUBJECT=jurisdiction`
を用いた `PROC GLIMMIX` は、母集団レベルの固定効果（年トレンド、
FATCA によるシフト）と管轄地間の分散成分の双方を算出します。
ランダム切片の分散は、*共通の時間トレンドを考慮した後で、設立率が
管轄地間でどれだけ異なるか* を教えてくれます。これはオフショア
金融センター母集団の構造的な異質性のスコアです。

In [ ]:
/* パネルデータセット: 1990〜2014 年の 管轄地 × 年 のセル。 */
処理 gql conn=オフショア out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* 年平均件数が 5 以上の管轄地を残します。 */
処理 sql;
    create table s20_keep_jur as
    選択 jurisdiction, avg(n) as avg_n
    from s20_raw
    group 基準 jurisdiction
    having avg(n) >= 5;
quit;

処理 sql;
    create table s20 as
    選択 r.*,
           r.year - 2002 as year_c,
           case 場合 r.year >= 2014 なら 1 他 0 終了 as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

処理 度数 データ=s20;
    tables jurisdiction;
    表題 "Section 20 — 管轄地一覧 retained in the panel";
実行;

/* 混合効果ポアソン GLMM: 固定効果として年トレンド + FATCA ステップ、
   管轄地ごとのランダム切片。 */
処理 glimmix データ=s20;
    分類 jurisdiction;
    模型 n = year_c fatca / dist=poisson link=log solution;
    乱数 intercept / subject=jurisdiction;
    表題 "Section 20 — mixed Poisson formation-rate model";
実行;

/* ランダム切片の効果を順位付けすれば、共通トレンドを体系的に
   上回る／下回る管轄地が浮かび上がります。PROC GLIMMIX の SOLUTION
   ステートメントは上の既定出力にこれらを表示するため、順位付けは
   読者に委ねます。 */

In [ ]:
/* レポート PDF を閉じ、Neo4j ライブラリを解放します。 */
ods pdf close;

図書館 オフショア clear;

## 再現性と出所

- **グラフデータソース:** ICIJ *Offshore Leaks* 研究データベースの
  *Paradise Papers* データセット。初公開は 2017 年 11 月。
  <https://offshoreleaks.icij.org/pages/database> で入手可能。
  上流の公開ダンプ
  `github.com/neo4j-graph-examples/icij-paradise-papers` を介して、
  プラットフォーム共有の `step-neo4j` サービス
  （Neo4j 5.26 Community Edition、サーバーレベルの読み取り専用）に
  読み込まれています。
- **OFAC SDN データ:** 米国財務省 OFAC 特別指定国民の公開 CSV
  エクスポート。2026 年 4 月に財務省の公開プレビュー API から取得。
  `data/ofac_sdn.csv` ファイルには、エントリ数で最大の 5 つの
  OFAC プログラムにわたってキュレーションした 500 行が含まれます。
  第 6 節の 2 段階スクリーンに使用。
- **OpenSanctions データ:** OpenSanctions *default* 統合データセットの
  2026-04-17 スナップショット。
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`
  からダウンロード。コミット済みファイル
  `data/opensanctions_default.csv` には、主要データセットが OFAC SDN、
  英国 HM Treasury、EU 金融制裁、国連安全保障理事会、カナダ、
  ベルギー、オーストラリア、スイス、その他各国の統合制裁リストの
  いずれかである 18,654 件の Person・Company スキーマの行が含まれます。
  ファイルを 2 MB 未満に保つため、別名は除外しました。
  ライセンス: ODbL 1.0（OpenSanctions）。第 14 節のエンリッチメントに使用。
- **租税回避地の順位:** Tax Justice Network *Corporate Tax Haven
  Index 2021* の公表順位。`https://cthi.taxjustice.net` の指数ランディング
  ページと、2021 年 3 月のプレスリリース
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`
  から編纂。コミット済みファイル `data/tax_haven_ranks.csv` は、
  パラダイス文書に登場する管轄地を、その CTHI 順位と派生した
  `[0, 1]` のリスク重みとともに列挙します。ライセンス: CC BY-SA
  4.0（Tax Justice Network）。第 15 節の総合ランキングに使用。
- **グラフアルゴリズム:** Louvain によるコミュニティ検出と固有
  ベクトル中心性（PageRank に最も近い自前の代替指標）は、読み取り
  専用の Cypher で取得した辺リストに対して `PROC NETWORK` が
  プロセス内で計算します。プラットフォーム共有の `step-neo4j`
  サービスはサーバーレベルの読み取り専用であるため、すべての
  グラフアルゴリズムは Neo4j Graph Data Science の書き込み操作では
  なく、Workspace ポッド内で実行されます。
- **統計手法:** `PROC LIFETEST` は Kaplan-Meier 推定量を、ログランク、
  Wilcoxon、Tarone-Ware の層別検定とともに使用します。`PROC PHREG` は
  Python/statsmodels ラッパーを用いて Breslow の同時発生処理により
  Cox 比例ハザードモデルを当てはめます。`PROC LOGISTIC` は最尤法に
  よる二項ロジスティック回帰を当てはめます。
- **リスク合成手法:** 第 11 節の総合影響力スコアは、次数、
  log-PageRank、管轄地の広さ、在任年数を `[0, 1]` に正規化し、
  固定重み（30/30/20/20）で合算します。第 15 節の総合法人リスク
  スコアは、上限を設けた役員数、log-PageRank、CTHI リスク重み、
  そして二値の制裁対象役員フラグを正規化し、それぞれ 0.25 の
  等しい重みで合算します。

分析全体は、このフォルダー内のスモークテスト用スクリプト
`./smoke.jenner` から再現できます。共有 `step-neo4j` サービスに対して
（`JENNER_NEO4J_HOST` と `JENNER_NEO4J_PASS` を設定した上で。これは
どの Workspace ポッドでもプラットフォームが行ってくれます）
エンドツーエンドで実行するとおよそ 2 分かかり、既存のパイプラインと
並んで追加された 5 つの新しい節を含め、すべてのクエリとすべての
PROC が、実際の 163,435 ノードのグラフ上で期待される出力を返すことを
検証します。